# Gun Violence Twitter Data Pipeline

This notebook consolidates the entire data processing pipeline: configuration, bounding box collection, filtering by locations/dates, and keyword filtering.

## 1. Configuration & Setup

In [ ]:
import os
import re
import json
import gzip
import glob
import pandas as pd
from multiprocessing import Pool
from datetime import timedelta
from tqdm.notebook import tqdm
from geopy.geocoders import Nominatim

# ==========================================
# FILE PATHS
# ==========================================
RAW_DATA_DIR = "/n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/"
OUTPUT_DIR = "../data/processed/"
BBOX_FILE = "../data/bboxes.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# INCIDENTS & DATES
# ==========================================
INCIDENTS = {
    "Uvalde, TX": "2022-05-24",
    "Highland Park, IL": "2022-07-04",
    "Monterey Park, CA": "2023-01-21",
    "Nashville, TN": "2023-03-27"
}

# ==========================================
# KEYWORDS & HASHTAGS
# ==========================================
HASHTAGS = [
    "#GunViolence", "#GunControl", "#GunReformNow", "#BanAssaultWeaponsNow", "#Parkland",
    "#SecondAmendment", "#2A", "#2ndAmendment", "#gunsafety", "#GunControlNow", "#ConcealedCarry",
    "#GunRights", "#NeverAgain", "#NRATerrorism", "#ProGun", "#NeverAlone", "#MarchForOurLives",
    "#OpenCarry", "#gunowners", "#GunDebate", "#FloridaShooting", "#massshooting",
    "#MentalHealthAwareness", "#SuicidePrevention", "#EndGunViolence", "#GunSense",
    "#GunSafetyNow", "#DisarmHate", "#StopGunViolence", "#GunViolencePrevention",
    "#SaveOurKids", "#EnoughIsEnough", "#StopTheViolence"
]

KEYWORDS = [
    "gun violence", "mass shooting", "gun control", "gun deaths", "firearm fatalities",
    "shooting incident", "accidental shooting", "gun accident", "firearm ownership",
    "assault weapons", "gun policy", "active shooter", "gun rally", "shotgun", "rifle",
    "handgun", "automatic weapons", "concealed weapon", "mental health crisis",
    "suicide by firearm", "accidental death", "gun reform", "firearm injuries",
    "gun ban", "shooting prevention", "gun legislation", "open carry", "self-defense shooting",
    "background checks", "anger", "fear", "pain", "grief", "advocacy", "hate", "sadness", 
    "rage", "blame", "cause", "responsibility", "Parkland shooting", "Sandy Hook", 
    "Las Vegas shooting", "Orlando nightclub shooting", "school shooting", "massacre",
    "income inequality", "education level", "poverty", "unemployment", "community violence",
    "neighborhood safety", "urban violence", "race and gun violence", "social isolation",
    "gang violence", "threat", "warning signs", "premeditated", "manifesto", "violent intentions",
    "planned attack", "threat assessment", "legislation", "gun registration", 
    "universal background checks", "red flag laws", "gun safety legislation", "gun lobby", 
    "firearm restriction", "buyback program", "law enforcement", "community policing", 
    "violence prevention", "intervention", "risk assessment", "emergency response", 
    "public safety", "crisis intervention", "strap", "heater", "gat", "piece", "pop off", 
    "clap back", "drill", "spin the block", "active shooter drill", "lockdown", "gun show", 
    "ammo shortage", "arms dealer", "illegal firearms", "3D-printed gun", "ghost gun"
]

hashtag_words = [h.replace("#", "").lower() for h in HASHTAGS]
all_phrases = list(set([k.lower() for k in KEYWORDS] + hashtag_words))

escaped_phrases = [re.escape(phrase) for phrase in all_phrases]
REGEX_PATTERN = re.compile(r"\b(?:" + "|".join(escaped_phrases) + r")\b", flags=re.IGNORECASE)


## 2. Collect Bounding Boxes

In [ ]:
def collect_bounding_boxes(locations, output_file=BBOX_FILE):
    geolocator = Nominatim(user_agent="gun_violence_research_cga")
    bboxes = {}
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    for loc in locations:
        try:
            location = geolocator.geocode(loc)
            if location and hasattr(location, 'raw') and 'boundingbox' in location.raw:
                bbox = location.raw['boundingbox']
                bboxes[loc] = {
                    "lat_min": float(bbox[0]),
                    "lat_max": float(bbox[1]),
                    "lon_min": float(bbox[2]),
                    "lon_max": float(bbox[3])
                }
                print(f"✅ Successfully found bounding box for {loc}")
            else:
                print(f"⚠️ Could not find bounding box for {loc}")
        except Exception as e:
            print(f"Error fetching data for {loc}: {e}")

    with open(output_file, "w") as f:
        json.dump(bboxes, f, indent=4)
    print(f"\nSaved bounding boxes to {output_file}")
    return bboxes

# Run bounding box collection
BBOXES = collect_bounding_boxes(list(INCIDENTS.keys()))


## 3. Location and Date Filtering

In [ ]:
# Precompute date ranges for each incident (t-2 to t+2 days)
INCIDENT_DATE_RANGES = {}
for city, date_str in INCIDENTS.items():
    incident_date = pd.to_datetime(date_str)
    start_date = incident_date - timedelta(days=2)
    end_date = incident_date + timedelta(days=2)
    INCIDENT_DATE_RANGES[city] = {
        "start": start_date,
        "end": end_date,
        "bbox": BBOXES.get(city)
    }

def process_file(file_path):
    """
    Reads a gzipped CSV of tweets and filters them by date and location.
    Assumes tab-separated values.
    """
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            df = pd.read_csv(f, sep='\t', on_bad_lines="skip", dtype=str)

        if 'tweet_date' not in df.columns or 'latitude' not in df.columns or 'longitude' not in df.columns:
            return pd.DataFrame()

        df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
        df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
        df['tweet_date'] = pd.to_datetime(df['tweet_date'], errors='coerce')
        df = df.dropna(subset=['tweet_date', 'latitude', 'longitude', 'tweet_text'])

        matched_tweets = []
        for city, conditions in INCIDENT_DATE_RANGES.items():
            start_date = conditions['start']
            end_date = conditions['end']
            bbox = conditions['bbox']
            
            if not bbox:
                continue

            date_mask = (df['tweet_date'] >= start_date) & (df['tweet_date'] <= end_date)
            lat_mask = (df['latitude'] >= bbox['lat_min']) & (df['latitude'] <= bbox['lat_max'])
            lon_mask = (df['longitude'] >= bbox['lon_min']) & (df['longitude'] <= bbox['lon_max'])
            
            city_df = df[date_mask & lat_mask & lon_mask].copy()
            if not city_df.empty:
                city_df['matched_incident'] = city
                matched_tweets.append(city_df)

        if matched_tweets:
            return pd.concat(matched_tweets, ignore_index=True)
        else:
            return pd.DataFrame()
    except Exception as e:
        # print(f"⚠️ Error in {os.path.basename(file_path)}: {e}")
        return pd.DataFrame()

def filter_locations_dates():
    print("Starting location and date filtering...")
    all_files = glob.glob(os.path.join(RAW_DATA_DIR, "**", "*.csv.gz"), recursive=True)
    
    if not all_files:
        print(f"No files found in {RAW_DATA_DIR}")
        return None

    NUM_CORES = min(20, os.cpu_count() or 1)
    print(f"Processing {len(all_files)} files using {NUM_CORES} cores...")

    with Pool(processes=NUM_CORES) as pool:
        results = list(tqdm(pool.imap_unordered(process_file, all_files), total=len(all_files)))

    results = [df for df in results if not df.empty]
    
    if results:
        final_df = pd.concat(results, ignore_index=True)
        out_path = os.path.join(OUTPUT_DIR, "location_date_filtered_tweets.csv.gz")
        final_df.to_csv(out_path, index=False, compression="gzip", sep='\t')
        print(f"✅ Saved {len(final_df)} tweets to {out_path}")
        return out_path
    else:
        print("No tweets matched the criteria.")
        return None


## 4. Keyword Filtering

In [ ]:
def filter_by_keywords():
    input_file = os.path.join(OUTPUT_DIR, "location_date_filtered_tweets.csv.gz")
    output_file = os.path.join(OUTPUT_DIR, "final_filtered_tweets.csv.gz")

    if not os.path.exists(input_file):
        print(f"Input file {input_file} not found.")
        return

    print(f"Loading {input_file}...")
    df = pd.read_csv(input_file, compression='gzip', sep='\t', dtype=str)
    
    if 'tweet_text' not in df.columns:
        print("Error: 'tweet_text' column not found in dataset.")
        return

    print(f"Total tweets before keyword filtering: {len(df)}")
    df = df.dropna(subset=['tweet_text'])
    
    print("Filtering Keywords...")
    mask = df['tweet_text'].str.contains(REGEX_PATTERN, na=False, regex=True)
    final_df = df[mask].copy()
    
    print(f"Total tweets after keyword filtering: {len(final_df)}")
    final_df.to_csv(output_file, index=False, compression='gzip', sep='\t')
    print(f"✅ Saved final filtered dataset to {output_file}")
